# Setup - Drive folder structure

**Run this once, in Colab, before anything else.**

Creates `MyDrive/afs-data` with the exact directory names `afs.paths.Paths`
expects, plus a README in every folder explaining what belongs there.

Safe to re-run. It never overwrites data - only refreshes the README files.

---
Repo: https://github.com/sandesh20lamichhane/android-feature-sufficiency

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Create the structure

`DATA_ROOT` must stay exactly this. The Colab bootstrap and `afs.paths` both
resolve against it, and renaming a folder breaks path resolution silently.

In [ ]:
from pathlib import Path

DATA_ROOT = Path('/content/drive/MyDrive/afs-data')

DIRS = [
    'raw/drebin', 'raw/cicmaldroid', 'raw/androzoo',
    'interim', 'processed', 'runs', 'records', 'cache', '_archive',
]

for d in DIRS:
    (DATA_ROOT / d).mkdir(parents=True, exist_ok=True)

print(f'created {len(DIRS)} directories under {DATA_ROOT}')

## 3. Write the READMEs

Each folder documents its own rules, so you don't have to remember them.

In [ ]:
READMES = {'README.md': '# afs-data - Drive storage for the Android feature-sufficiency study\n\nThis tree is the ARTIFACT STORE. It pairs with the code repo\n(android-feature-sufficiency) but is never committed to it.\n\nLocation must be exactly: MyDrive/afs-data\nThe Colab bootstrap sets AFS_DATA_ROOT to this path and afs.paths.Paths\nexpects these exact directory names. Renaming any of them breaks path\nresolution silently.\n\n## The split that matters\n\n  Feature matrices, models, shards ....... Drive only, never git\n  Raw datasets ........................... Drive only, never git\n  metrics.json, config.json, manifests ... Drive AND mirrored to git\n  Decision records, lab notebook ......... git is source of truth\n\nArtifacts are large and regenerable from code plus config. Records are small,\ntextual, and NOT regenerable - they are what tells you in six months what ran,\non which commit, with which config, and what came out.\n\nAfter each run:\n    python scripts/sync_records.py <run_id>\n    git add -A && git commit -m "run <run_id>" && git push\n\n## Directories\n\n  raw/        Downloaded datasets. WRITE-ONCE. Every stage reads from here and\n              writes elsewhere. Corrupting raw data mid-project is the one\n              unrecoverable failure.\n  interim/    Parsed but not yet featurised. Regenerable, safe to delete.\n  processed/  Feature matrices, vocabularies, split indices.\n  runs/       One directory per experiment run. Created automatically.\n              Never reused: new config means new run id.\n  records/    Drive-side mirror of the repo\'s docs/. Read-only convenience copy.\n  cache/      Deletable at any time without loss.\n  _archive/   Superseded runs. MOVE here, NEVER delete.\n\n## Run directory layout\n\n  runs/20260904-a1b2c3d4e5-1f2e3d4c/\n    config.json         resolved config, exactly as hashed\n    data_summary.json   counts, date range, provenance breakdown\n    metrics.json        results  <- mirror into git\n    checkpoints/        models + .manifest.json sidecars\n    partial/            resumable shards, part-00000.parquet ...\n    logs/\n\nRun id = YYYYMMDD-<git sha>-<config hash>\n\n## Do not put secrets here\n\nThe AndroZoo API key does NOT belong in Drive. It goes in Colab Secrets:\nuserdata.get(\'ANDROZOO_KEY\'). Same for the GitHub token. Anything in Drive is\none bad share-link away from public.\n', 'raw/README.md': '# raw/ - WRITE ONCE\n\nAfter download, nothing here is edited, cleaned, renamed, or reorganised.\nEvery pipeline stage reads from here and writes to interim/ or processed/.\n\n## Integrity\n\nHash everything at download; verify before each stage:\n\n    python scripts/hash_raw.py      # writes docs/raw_checksums.json -> commit it\n\nSilent bit-rot is rare. A partially-downloaded dataset that you then train on\nfor three weeks is the failure this actually prevents.\n\n## Backup\n\nArtifacts elsewhere in afs-data/ are regenerable, so they need no backup.\nraw/ does. For public datasets, the download scripts plus checksums in the repo\nare a sufficient recipe. The exception is anything obtained under a data-use\nagreement - if AndroZoo access lapses or a mirror disappears, a recipe does not\nhelp. Keep a second physical copy of those.\n', 'raw/drebin/PLACE_FILES_HERE.md': '# raw/drebin/\n\n## Expected layout\n\n  drebin/\n    feature_vectors/     ~5,560 files, one per app, named by sha256\n                         plain text, one "type::value" per line\n    sha256_family.csv    sha256,family  (179 families)\n\nFeature-vector lines look like permission::android.permission.SEND_SMS,\napi_call::getDeviceId, intent::android.intent.action.BOOT_COMPLETED. The loader\nmaps these onto the perm:: / intent:: / api:: prefixes that afs.features.sets\nselects on.\n\n## Access\n\nRequest from the TU Braunschweig / Drebin project page. Samples are distributed\nunder a research agreement - read the terms before redistributing anything\nderived from them.\n\n## WARNING: Drebin ships MALWARE ONLY\n\nThere is no benign corpus in this release. The original paper used ~123k benign\napps that are not distributed. Whatever benign set you choose silently\ndetermines your result, which is why cross-paper comparison on "Drebin" is\nlargely meaningless.\n\nThis is the single most attackable choice in the study. Fill in\ndocs/decisions/0002-drebin-benign-corpus.md BEFORE implementing the loader.\nPlan of record: date- and market-matched AndroZoo apps from Aug 2010 - Oct 2012.\n\n## Era caveat for the paper\n\nDrebin\'s malware predates Android 6.0 runtime permissions (Oct 2015). In that\nera malware was grotesquely over-permissioned - SEND_SMS + RECEIVE_SMS +\nREAD_SMS together is close to a label. Any "permissions are sufficient" result\nhere is an era artifact until shown otherwise on post-2015 data.\n', 'raw/cicmaldroid/PLACE_FILES_HERE.md': '# raw/cicmaldroid/\n\n## Expected layout\n\n  cicmaldroid/\n    feature_vectors_static.csv     ~470 static features (permissions, intents)\n    feature_vectors_syscalls.csv   CopperDroid dynamic features\n    labels.csv                     sample hash -> class\n\nJoined on sample hash by the loader. Static columns get perm:: / intent::\nprefixes; dynamic columns get syscall:: / binder:: / composite::.\n\n## Why this dataset earns its place\n\nIt is the only one of the three that ships BOTH static and dynamic features for\nthe SAME samples. That gives a clean permissions-vs-behaviour contrast without\nchanging corpora - otherwise the confound that ruins this comparison.\n\nCollected Dec 2017 - Dec 2018, so entirely post-runtime-permissions.\n\n## WARNING: two biases, both favouring the permissions arm\n\nRiskware. A labelled class here, and riskware is definitionally\nover-permissioned software. Including it lets you "confirm" the premise for the\nwrong reason. Report every result twice, with and without.\n\nSandbox attrition. Not all samples executed successfully under CopperDroid, so\nthe dynamic subset is smaller and non-randomly so. Samples that detect or crash\nthe sandbox are plausibly the more sophisticated ones - precisely the population\nwhere deep features should matter most. Characterise the attrition rate by class\nrather than silently dropping it.\n\nSee docs/decisions/0003-cicmaldroid-riskware.md\n\n## Access\n\nCanadian Institute for Cybersecurity, UNB. Public download, no agreement needed.\n', 'raw/androzoo/PLACE_FILES_HERE.md': "# raw/androzoo/\n\n## REQUEST THE API KEY NOW\n\nApproval needs an academic email and takes DAYS TO WEEKS. It blocks two separate\nthings, so it is the first task regardless of which axis leads the paper:\n\n  1. The temporal axis - AndroZoo supplies dex_date for real past/future\n     ordering. Neither Drebin (2010-12) nor CICMalDroid (2017-18) spans enough\n     time alone.\n  2. The Drebin benign corpus - a benign set with KNOWN provenance, date- and\n     market-matched, rather than whatever a prior reproduction happened to use.\n\nRequest at the AndroZoo site (University of Luxembourg).\n\n## Expected layout\n\n  androzoo/\n    latest.csv.gz    full index: sha256, dex_date, markets, vt_detection, ...\n    apks/            only the subset you actually pull\n\nThe index is large; download once and treat as immutable.\n\n## Key handling\n\nThe API key does NOT go in this folder or anywhere in Drive. Put it in Colab\nSecrets and read it with userdata.get('ANDROZOO_KEY').\n\n## Open methodological questions\n\nBoth must be settled in docs/decisions/0004-androzoo-temporal.md before use:\n\n  - Gap length between train_end and test_start. A gap prevents near-duplicate\n    repackagings straddling the boundary from leaking the test set into\n    training. Setting it defensibly needs a near-duplicate analysis first.\n  - dex_date reliability. Some samples carry forged or missing compile\n    timestamps. Needs a documented fallback and exclusion rule, since a silently\n    mis-dated sample corrupts exactly the protocol it appears in.\n\n## Benign labelling\n\nvt_detection is the usual threshold field. State your cutoff explicitly (0 for\nbenign is stricter than <=3) and report it - reviewers will ask, and the choice\nmoves the numbers.\n", 'interim/README.md': '# interim/\n\nParsed but not yet featurised: manifest extractions, joined tables, decoded\nsandbox traces. Regenerable from raw/ - safe to delete if space runs short.\nWritten by the pipeline, not by hand.\n', 'processed/README.md': '# processed/\n\nFeature matrices, vocabularies, and split index files.\n\nEach artifact carries a .manifest.json sidecar recording the config hash, git\nSHA, seed, and library versions that produced it. Filenames embed a hash of\n(stage, config, inputs), so changing a config value produces a DIFFERENT FILE\nrather than a silent reuse of a stale one.\n\nNever rename anything here. The name is the cache key.\n\n## Vocabulary leakage\n\nFeature vocabularies are fit on the TRAINING SPLIT ONLY (fit_vocab_on: train in\nconfigs/features/nested.yaml). Fitting on the full dataset leaks test\ninformation and is one of the standard errors this study exists to criticise -\nso getting it wrong here would be fatal.\n', 'runs/README.md': '# runs/\n\nOne directory per experiment run, created automatically by Paths.run_dir().\n\n  runs/20260904-a1b2c3d4e5-1f2e3d4c/\n    config.json         resolved config, exactly as hashed\n    data_summary.json   counts, date range, provenance breakdown\n    metrics.json        results  <- mirror into git\n    checkpoints/        models + manifest sidecars\n    partial/            resumable shards, part-00000.parquet ...\n    logs/\n\nRun id = YYYYMMDD-<git sha>-<config hash>\n\n## Rules\n\nNever reuse a run id. New config means new run. Disk is cheaper than a result\nyou cannot trace back.\n\nOne Colab session per run id. Two sessions writing the same directory produce\n"file (1).parquet" collisions and silent corruption. For parallelism, use\ndifferent run ids.\n\nCommit before any run you intend to cite. The pipeline warns on a dirty working\ntree, because the recorded SHA then does not describe the code that produced\nthe result.\n\nArchive, do not delete:\n    python scripts/archive_run.py <run_id>\n\n## Resuming\n\npartial/ holds sharded parquet from ResumableLoop. On restart the loop reads\nwhich item ids are already done and skips them, so a disconnect costs one item\nrather than the whole run. Do not delete these mid-experiment.\n', 'records/README.md': "# records/\n\nDrive-side MIRROR of the repo's docs/ - decision records, lab notebook, raw\nchecksums.\n\n## Direction is one-way\n\nGit is the source of truth. This is a read-only convenience copy.\n\nNever edit here and let changes flow back. Drive has no diff history and no\natomic commits; reconstructing a methodology from Drive version history does not\nwork. For a paper whose contribution is that other people's evaluations were\nsloppy, an intact audit trail is not optional.\n\nEdit in the repo, commit, then re-mirror.\n", 'cache/README.md': '# cache/\n\nDeletable at any time without loss. Nothing here is an input to any result.\nIf Drive quota gets tight, this goes first, then interim/.\n', '_archive/README.md': '# _archive/\n\nSuperseded runs. MOVE here - NEVER delete.\n\nYou will be tempted to clean up after a run is replaced. Two months later you\nwill want to know what the old numbers were, usually because a reviewer asks\nwhy a figure changed between drafts.\n\n    python scripts/archive_run.py 20260904-a1b2c3d4e5-1f2e3d4c\n'}

In [ ]:
for rel, text in READMES.items():
    p = DATA_ROOT / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text)
print(f'wrote {len(READMES)} README files')

## 4. Verify

Confirms the tree matches what the code expects and reports free space.
Feature matrices across three datasets and four feature sets, plus per-sample
evasion shards, run to tens of GB - and Drive quota is shared with everything
else on the account.

In [ ]:
import shutil

EXPECTED = ['raw', 'interim', 'processed', 'runs', 'records', 'cache', '_archive']
ok = True
for name in EXPECTED:
    exists = (DATA_ROOT / name).is_dir()
    ok &= exists
    print(f"  {'OK  ' if exists else 'MISS'} {name}")

free_gb = shutil.disk_usage('/content/drive/MyDrive').free / 1e9
print(f'\nfree space: {free_gb:.1f} GB')
if free_gb < 20:
    print('  WARN  under 20 GB - matrices + evasion shards run to tens of GB')

print('\nSTRUCTURE OK' if ok else '\nSOMETHING MISSING - re-run cell 2')

### Full tree

In [ ]:
for p in sorted(DATA_ROOT.rglob('*')):
    depth = len(p.relative_to(DATA_ROOT).parts) - 1
    print('  ' * depth + ('[D] ' if p.is_dir() else '    ') + p.name)

## Next steps

1. **Request the AndroZoo API key today.** Academic email required, approval
   takes days to weeks. It blocks both the temporal axis and the Drebin benign
   corpus, so nothing else on the critical path can start without it.

2. Read `raw/drebin/PLACE_FILES_HERE.md`. Drebin ships malware only - the benign
   corpus you pick silently determines your result, and it is the first thing a
   reviewer will attack.

3. Open `notebooks/00_setup_verify.ipynb` in the repo to clone, install, and
   verify the pipeline end to end.

**Do not put the AndroZoo key in Drive.** Use Colab Secrets:
`userdata.get('ANDROZOO_KEY')`. Anything in Drive is one bad share-link from
public.